# Natural language to SQL

**Run in [Google Colab](https://colab.research.google.com/) For GPU.**

This model have  Mistral as a base and it has been fine-tuned to excel in SQL code generation.

In [3]:


from google.colab import userdata
token = userdata.get('HF_TOKEN')




In [ ]:
# from google.colab import userdata
# userdata.get('HF_TOKEN')

In [4]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/accelerate.git to /tmp/pip-req-build-qq752br2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-qq752br2
  Resolved https://github.com/huggingface/accelerate.git to commit 665444ceb62211f2b410d0d0fdb4bc013c5effdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for accelerate: filename=accelerate-1.15.0.dev0-py3-none-any.whl size=390066 sha256=152422b6639f0fc6b3dc0fb74410ce4958e8c98fc40e45f8682427abf81b4d34
  Stored in directory: /tmp/pip-ephem-wheel-cache-8rwncq_t/wheels/5a/20/fb/1221fe933b56fe7ac69fd00159d9a1950bc8ced38198abc18f
Successfully built accelerate
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [6]:
model_name = "defog/sqlcoder-7b"

We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [7]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [8]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

This function wraps the call to *model.generate*

In [10]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [11]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE users (
    user_id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100),
    signup_date DATE
);

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100),
    price DECIMAL(10, 2),
    stock INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    user_id INT,
    product_id INT,
    quantity INT,
    order_date DATE,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);


    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """

In [12]:
sp_nl2sql = sp_nl2sql.format(question="YHow many orders did user John place")
print(sp_nl2sql)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE users (
    user_id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100),
    signup_date DATE
);

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100),
    price DECIMAL(10, 2),
    stock INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    user_id INT,
    product_id INT,
    quantity INT,
    order_date DATE,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);


    ### Response
    Based on your instructions, here is the SQL query I have generated to answer t

In [13]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [14]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [15]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT COUNT(order_id) AS number_of_orders FROM orders WHERE user_id IN (SELECT user_id FROM users WHERE username ilike '%John%');


The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [16]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

   YOUR TABLES HERE

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `{question}`: How many users signed up in 2023?
- SQL Query:
```sql
SELECT COUNT(*) FROM users WHERE signup_date >= '2023-01-01';
    ```sql3
    """


In [17]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

   YOUR TABLES HERE

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `Return The name of the best paid employee`: How many users signed up in 2023?
- SQL Query:
```sql
SELECT COUNT(*) FROM users WHERE signup_date >= '2023-01-01';
    ```sql3
    


In [18]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

KeyboardInterrupt: 

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [23]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE

    ### Samples

    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """


In [24]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE

    ### Samples

    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `Return The name of the best paid employee`:
    ```sql3
    


In [25]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [20]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT COUNT(order_id) AS number_of_orders FROM orders WHERE user_id IN (SELECT user_id FROM users WHERE username ilike '%John%');


#Now the question in spanish.


In [26]:
sp_nl2sql3 = sp_nl2sql3b.format(question="¿Cuántos usuarios se registraron en 2023?")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE

    ### Samples

    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `¿Cuántos usuarios se registraron en 2023?`:
    ```sql3
    


In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT COUNT(*) FROM users WHERE to_date(created_at, 'YYYY-MM-DD') BETWEEN to_date('2023-01-01', 'YYYY-MM-DD') AND to_date('2023-12-31', 'YYYY-MM-DD');


The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [27]:
database_schema = """
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    hire_date DATE,
    salary DECIMAL(10, 2)
);

CREATE TABLE departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(50),
    location VARCHAR(50)
);

ALTER TABLE employees ADD FOREIGN KEY (department) REFERENCES departments(department_name);
"""

Variation 1: Concise Instructions with Explicit Schema


# Variation 1: Concise Instructions with Explicit Schema

In [28]:

sp_nl2sql_ex1 = """
### Instructions:
Generate a SQL query to answer the user's question based on the provided schema.

### Schema:
{schema}

### Question:
{question}

### SQL Query:
```sql
"""""

#Variation 2: More Detailed Instructions and Schema Integrated

In [ ]:
sp_nl2sql_ex2 = """
Your task is to write a SQL query given a natural language question and the database schema. The database has the following structure:

{schema}

Please provide the SQL query that answers the following question:

{question}

```sql
"""



# Variation 3: Focusing on Table and Column Names

In [ ]:
sp_nl2sql_ex3 = """
Convert the following question into a SQL query.
Consider the available tables and columns:
- Table: employees (employee_id, name, department, hire_date, salary)
- Table: departments (department_id, department_name, location)

Question: {question}

SQL:
```sql
"""

In [ ]:
question_ex1 = "Encuentra los nombres de los empleados en el departamento de 'Ventas'."
question_ex2 = "Calcula el salario promedio de todos los empleados."
question_ex3 = "Lista los empleados que fueron contratados después de 2022."

In [ ]:
print("--- Testing Variation 1 ---")
prompt_v1 = sp_nl2sql_ex1.format(schema=database_schema, question=question_ex1)

# Tokenize and move to GPU
input_v1 = tokenizer(prompt_v1, return_tensors="pt").to('cuda')

# Run inference (make sure `foundation_model` is your loaded model)
response_v1 = get_outputs(foundation_model, input_v1)

# Decode output
sql_v1 = tokenizer.batch_decode(response_v1, skip_special_tokens=True)

# Extract just the SQL part
print(sql_v1[0].split("```sql")[-1].split("```")[0].strip() + ";")

--- Testing Variation 1 ---
SELECT employees.employee_id, employees.name, employees.department, employees.salary FROM employees WHERE employees.department ILIKE '%Ventas%' ORDER BY employees.employee_id NULLS LAST;;


In [ ]:
print("\n--- Testing Variation 2 ---")

# Fill in prompt
prompt_v2 = sp_nl2sql_ex2.format(schema=database_schema, question=question_ex2)

# Tokenize and send to GPU
input_v2 = tokenizer(prompt_v2, return_tensors="pt").to('cuda')

# Generate output
response_v2 = get_outputs(foundation_model, input_v2)

# Decode the generated text
sql_v2 = tokenizer.batch_decode(response_v2, skip_special_tokens=True)

# Extract SQL from output using markdown-style marker
print(sql_v2[0].split("```sql")[-1].split("```")[0].strip() + ";")


--- Testing Variation 2 ---
QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees;
 SELECT AVG(employees.salary) AS average_salary FROM employees; QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /******/ QUERY: SELECT AVG(employees.salary) AS average_salary FROM employees; /***

In [ ]:
print("\n--- Testing Variation 3 ---")

# Format the prompt
prompt_v3 = sp_nl2sql_ex3.format(question=question_ex3)  # question_ex3 = "List the employees hired after 2022."

# Tokenize and run
input_v3 = tokenizer(prompt_v3, return_tensors="pt").to('cuda')
response_v3 = get_outputs(foundation_model, input_v3)

# Decode and clean the output
sql_v3 = tokenizer.batch_decode(response_v3, skip_special_tokens=True)
torch.cuda.empty_cache()

# Extract SQL from markdown block
print(sql_v3[0].split("```sql")[-1].split("```")[0].strip() + ";")



--- Testing Variation 3 ---
SELECT e.employee_id, e.name, d.department_name, to_char(e.hire_date, 'YYYY') AS hire_year FROM employees e JOIN departments d ON e.department = d.department_id WHERE to_number(to_char(e.hire_date, 'YYYY'), '9999') > 2022 ORDER BY e.employee_id NULLS LAST;т
 SELECT e.employee_id, e.name, d.department_name, to_char(e.hire_date, 'YYYY') AS hire_year FROM employees e JOIN departments d ON e.department = d.department_id WHERE to_number(to_char(e.hire_date, 'YYYY'), '9999') > 2022 ORDER BY e.employee_id NULLS LAST;т
 SELECT e.employee_id, e.name, d.department_name, to_char(e.hire_date, 'YYYY') AS hire_year FROM employees e JOIN departments d ON e.department = d.department_id WHERE to_number(to_char(e.hire_date, 'YYYY'), '9999') > 2022 ORDER BY e.employee_id NULLS LAST;т
 SELECT e.employee_id, e.name, d.department_name, to_char(e.hire_date, 'YYYY') AS hire_year FROM employees e JOIN departments d ON e.department = d.department_id WHERE to_number(to_char(e.hire_da

Summary:
The notebook tests three prompt styles to convert natural language questions into SQL queries using the defog/sqlcoder-7b model and a given database schema:

    Variation 1: Clear instructions with full CREATE TABLE schema.

    Variation 2: Instructions with schema integrated into the prompt more naturally.

    Variation 3: Simplified schema listing tables and columns only.

Each variation uses different questions to evaluate how prompt structure affects SQL generation.

Key Insights:

    Prompt wording and schema presentation strongly impact model output quality.

    Full vs. simplified schema formats may influence accuracy depending on schema complexity.

    Handling multilingual inputs (like Spanish) may be limited without model fine-tuning.

    Extracting SQL from model output relies on consistent formatting (e.g., markdown code blocks).

    The code setup is modular and repeatable, good for further experiments.

Takeaway:
Careful prompt design is essential to get precise SQL from language models like SQLCoder.